In [1]:
# Temporal Aggregation 

import pandas as pd
import pyarrow.parquet as pq
import numpy as np
from tqdm import tqdm

print("Temporal Aggregation (Hourly + Weekday)")

taxi_path = r'I:\GEO DATA ANALYSIS\chicago_mobility\chicago_taxi_2024.parquet'


parquet_file = pq.ParquetFile(taxi_path)

chunksize = 500_000
temporal_agg = []

print(f"Processing taxi data in chunks of {chunksize:,} rows...")

for i, batch in enumerate(parquet_file.iter_batches(batch_size=chunksize, columns=['trip_start_timestamp', 'pickup_community_area'])):
    chunk = batch.to_pandas()
    
    # Convert timestamp and extract temporal features
    chunk['trip_start_timestamp'] = pd.to_datetime(chunk['trip_start_timestamp'])
    chunk['hour'] = chunk['trip_start_timestamp'].dt.hour
    chunk['day_of_week'] = chunk['trip_start_timestamp'].dt.day_name()
    chunk['is_weekend'] = chunk['trip_start_timestamp'].dt.weekday >= 5

    # Aggregate
    agg = chunk.groupby(['pickup_community_area', 'hour', 'day_of_week', 'is_weekend']).size().reset_index(name='trip_count')
    temporal_agg.append(agg)
    
    if (i + 1) % 10 == 0:
        print(f"Processed {i+1} chunks...")

# Combine all chunks
temporal_df = pd.concat(temporal_agg, ignore_index=True)

# Final aggregation (sum trip counts)
temporal_df = temporal_df.groupby(['pickup_community_area', 'hour', 'day_of_week', 'is_weekend'])['trip_count'].sum().reset_index()

print(f"\nTemporal aggregation completed!")
print(f"Total rows: {len(temporal_df):,}")
print(temporal_df.head())

# Save
temporal_df.to_parquet('chicago_taxi_temporal_agg.parquet', index=False)
print("Saved as 'chicago_taxi_temporal_agg.parquet'")

# Summary by hour
peak_summary = temporal_df.groupby('hour')['trip_count'].sum().reset_index()
print("\nTotal trips by hour of day (top 8):")
print(peak_summary.sort_values('trip_count', ascending=False).head(8))

Temporal Aggregation (Hourly + Weekday)
Processing taxi data in chunks of 500,000 rows...
Processed 10 chunks...
Processed 20 chunks...

Temporal aggregation completed!
Total rows: 12,873
   pickup_community_area  hour day_of_week  is_weekend  trip_count
0                    1.0     0      Friday       False         105
1                    1.0     0      Monday       False         148
2                    1.0     0    Saturday        True         133
3                    1.0     0      Sunday        True         195
4                    1.0     0    Thursday       False         108
Saved as 'chicago_taxi_temporal_agg.parquet'

Total trips by hour of day (top 8):
    hour  trip_count
17    17      994075
16    16      961621
15    15      938086
18    18      908506
14    14      905402
13    13      886612
12    12      866094
11    11      808397


In [2]:
# Temporal Predictive Model

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import gc

print("Temporal Predictive Model")

# Load data
temporal_df = pd.read_parquet('chicago_taxi_temporal_agg.parquet')

# Downcast aggressively
temporal_df['pickup_community_area'] = temporal_df['pickup_community_area'].fillna(0).astype('int16')
temporal_df['hour'] = temporal_df['hour'].astype('int8')
temporal_df['is_weekend'] = temporal_df['is_weekend'].astype('int8')
temporal_df['trip_count'] = temporal_df['trip_count'].astype('int32')

# Cyclical features
temporal_df['hour_sin'] = np.sin(2 * np.pi * temporal_df['hour'] / 24).astype('float32')
temporal_df['hour_cos'] = np.cos(2 * np.pi * temporal_df['hour'] / 24).astype('float32')

features = ['pickup_community_area', 'hour', 'is_weekend', 'hour_sin', 'hour_cos']
target = 'trip_count'


if len(temporal_df) > 5_000_000:
    print("Dataset very large. Sampling 50% to prevent Kernel death...")
    temporal_df = temporal_df.sample(frac=0.5, random_state=42)

X = temporal_df[features]
y = temporal_df[target]

del temporal_df
gc.collect()

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)


del X_train, y_train
gc.collect()

# Native XGBoost API
params = {
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'max_depth': 4,           
    'eta': 0.1,               
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'seed': 42
}

print("Training native XGBoost model...")
bst = xgb.train(
    params, 
    dtrain, 
    num_boost_round=300,
    evals=[(dtest, "test")],
    early_stopping_rounds=10,
    verbose_eval=50
)

# Evaluation
y_pred = bst.predict(dtest)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nModel Results:")
print(f"R² Score: {r2:.3f}")
print(f"MAE: {mae:,.0f} trips")

# Feature Importance
importance = bst.get_score(importance_type='gain')
importance_df = pd.DataFrame({
    'feature': list(importance.keys()),
    'importance': list(importance.values())
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(importance_df)

# Save
joblib.dump(bst, 'taxi_temporal_model_native.pkl')
print("Saved as 'taxi_temporal_model_native.pkl'")

Temporal Predictive Model
Training native XGBoost model...
[0]	test-rmse:3933.64103
[50]	test-rmse:1439.54042
[100]	test-rmse:1097.38513
[150]	test-rmse:983.13692
[200]	test-rmse:900.10569
[250]	test-rmse:876.04884
[299]	test-rmse:848.92135

Model Results:
R² Score: 0.954
MAE: 294 trips

Feature Importance:
                 feature   importance
0  pickup_community_area  331178080.0
4               hour_cos  116169528.0
3               hour_sin   69879712.0
1                   hour   64876252.0
2             is_weekend   23260948.0
Saved as 'taxi_temporal_model_native.pkl'
